# PLaTune

**PLaTune** (Pretrained Latents Tuner) adds time-varying musical controls on top of any pretrained generative model with an exposed latent space — without retraining the codec. [ISMIR 2025 paper](https://ismir2025.ismir.net/) & [source on GitHub](https://github.com/acids-ircam/platune).

This notebook is structured to mirror the RAVE Kaggle template (Miniconda environment, force-installed torch, save-output-as-dataset resume pattern). It runs the full pipeline:

1. Install a clean Miniconda env at `/kaggle/temp/miniconda` (avoids conflicts with Kaggle's preinstalled packages).
2. Clone PLaTune and install its requirements.
3. **Preprocess** the audio into an LMDB of codec latents + control attributes.
4. **Train** the rectified-flow transformer.
5. **Resume training** across Kaggle sessions by saving each run's output as a Dataset and attaching it back as input.

**Before running** — in the Kaggle sidebar:
- Accelerator → **GPU** (T4 / P100 / L4).
- Internet → **On** (needed for the conda/pip installs and the Music2Latent checkpoint download).

## Configuration

All paths and hyperparameters live here. Every cell below reads from these variables.

In [ ]:
import os

# --- environment -----------------------------------------------------------
MINICONDA = '/kaggle/temp/miniconda'
PYTHON    = f'{MINICONDA}/bin/python'
PIP       = f'{MINICONDA}/bin/pip'
CONDA     = f'{MINICONDA}/bin/conda'

# --- repo source -----------------------------------------------------------
REPO_URL    = 'https://github.com/skabbit/platune.git'
REPO_BRANCH = 'main'
# If you uploaded the platune repo as a Kaggle Dataset (offline kernels),
# set REPO_LOCAL to its mount path (e.g. '/kaggle/input/platune-source').
REPO_LOCAL  = None
REPO_PATH   = '/kaggle/working/platune'

# --- data ------------------------------------------------------------------
INPUT_AUDIO     = '/kaggle/input/your_audio_dataset'   # raw audio (only used in preprocess)
PROCESSED_LMDB  = '/kaggle/working/processed'          # LMDB output (or pre-built input)
EMB_MODEL       = 'music2latent'   # 'music2latent' or path to a torchscript codec
PARSER_NAME     = 'simple_parser'  # see platune/datasets/parsers.py
DATA_CONFIG     = 'medley_solos'   # gin file in platune/datasets/configs/
DESCRIPTORS     = ['rms', 'centroid', 'bandwidth', 'booming', 'sharpness',
                   'integrated_loudness', 'loudness1s']
NUM_SIGNAL      = 131072
SAMPLE_RATE     = 44100
PREP_BATCH      = 32
DB_SIZE_GB      = 64
USE_BASIC_PITCH = True
MIDI_ATTRS      = True

# --- training --------------------------------------------------------------
TRAIN_CONFIG = 'v1'                       # platune/configs/<TRAIN_CONFIG>.gin
RUN_NAME     = 'your_training_name'
SAVE_DIR     = '/kaggle/working/runs'
MAX_STEPS    = 50_000
VAL_EVERY    = 5_000
BUILD_CACHE  = False
BINS_VALUES_FILE = None     # 'bins_values.pkl' (mutually exclusive with MIN_MAX_FILE)
MIN_MAX_FILE     = None     # 'metadata_attributes.json'
LMDB_KEYS_FILE   = None     # 'lmdb_keys'

# --- resume ----------------------------------------------------------------
# Set to the input mount of a previous run's output dataset, e.g.
# '/kaggle/input/platune-run-v1' (which contains runs/<RUN_NAME>/last.ckpt).
RESUME_FROM_INPUT = None

os.makedirs('/kaggle/temp', exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
print('python    :', PYTHON)
print('repo path :', REPO_PATH)
print('lmdb path :', PROCESSED_LMDB)
print('save dir  :', os.path.join(SAVE_DIR, RUN_NAME))

## Install Miniconda, PLaTune, dependencies

Miniconda is installed under `/kaggle/temp/` (ephemeral — wiped at the end of the session, not saved to the notebook output). PLaTune's requirements.txt asks for Python ≥ 3.12, so we install the latest Miniconda (currently py3.12).

In [ ]:
# Install Miniconda
%cd /kaggle/temp
!curl -sL https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -o miniconda.sh
!chmod +x miniconda.sh
!sh miniconda.sh -b -p {MINICONDA}

# Verify Python ≥ 3.12 (required by PLaTune)
!{PYTHON} --version

# Upgrade kernel deps and install ffmpeg (audio backend for librosa)
!{PIP} install --upgrade -q ipython ipykernel
!{CONDA} install -y -q -c conda-forge ffmpeg

### Get the PLaTune source code

In [ ]:
import shutil, subprocess

if os.path.isdir(REPO_PATH):
    shutil.rmtree(REPO_PATH)

if REPO_LOCAL is not None:
    print(f'copying repo from {REPO_LOCAL} -> {REPO_PATH}')
    shutil.copytree(REPO_LOCAL, REPO_PATH)
else:
    print(f'cloning {REPO_URL} ({REPO_BRANCH})')
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, REPO_PATH])

!ls {REPO_PATH}

### Install PLaTune requirements

We install everything from `requirements.txt` **except** `nn_tilde` (only needed for `nn~` Max/PD export, has heavy C++ build deps that often fail on Kaggle). Then install the `platune` package itself in editable mode.

In [ ]:
# Strip nn_tilde from requirements (optional, not used at training time)
!grep -v '^nn_tilde' {REPO_PATH}/requirements.txt > /kaggle/temp/requirements.txt
!{PIP} install -q -r /kaggle/temp/requirements.txt

# Install the platune package itself
!{PIP} install -q -e {REPO_PATH}

In [ ]:
# Force-reinstall torch with the CUDA 11.8 build (matches PLaTune's pinned versions)
!{PIP} install --force-reinstall -q \
    torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 \
    --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Patch music2latent for PyTorch >= 2.6: its pretrained checkpoint contains numpy
# globals that the new weights_only=True default refuses to unpickle.
patch_script = '''
import importlib.util, pathlib, re
spec = importlib.util.find_spec("music2latent.inference")
if spec is None or spec.origin is None:
    raise SystemExit("music2latent not installed")
p = pathlib.Path(spec.origin)
src = p.read_text()
patched = re.sub(
    r"torch\\.load\\(self\\.load_path_inference,\\s*map_location=self\\.device\\)",
    "torch.load(self.load_path_inference, map_location=self.device, weights_only=False)",
    src,
)
p.write_text(patched)
print("patched" if patched != src else "already patched / no match", p)
'''
!{PYTHON} -c "{patch_script}"

In [ ]:
# Sanity-check the GPU is visible to the Miniconda Python
!{PYTHON} -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')"
!nvidia-smi -L

## Preprocess the dataset

Preprocessing is run once per training dataset. The output (codec latents + audio descriptors + MIDI-derived attributes) is written to `/kaggle/working/processed/` as an LMDB. Set `INPUT_AUDIO` in the Configuration cell to your audio dataset's mount path (typically `/kaggle/input/<your_audio_dataset>`).

***Disable this section once preprocessing is done — once the LMDB is in `/kaggle/working/processed/` (and saved as a Dataset for later sessions) you only need to run training.***

In [ ]:
%cd /kaggle/working
os.makedirs(PROCESSED_LMDB, exist_ok=True)

prep_cmd = [
    PYTHON, f'{REPO_PATH}/scripts/prepare_dataset.py',
    '-i', INPUT_AUDIO,
    '-o', PROCESSED_LMDB,
    '-s', str(DB_SIZE_GB),
    '-p', PARSER_NAME,
    '-c', DATA_CONFIG,
    '-m', EMB_MODEL,
    '--gpu', '0',
    '-n', str(NUM_SIGNAL),
    '--sr', str(SAMPLE_RATE),
    '-b', str(PREP_BATCH),
    '--cut_silences',
    '--save_waveform',
]
if USE_BASIC_PITCH: prep_cmd.append('--use_basic_pitch')
if MIDI_ATTRS:      prep_cmd.append('--midi_attributes')
for d in DESCRIPTORS: prep_cmd += ['-l', d]

print(' '.join(prep_cmd))
!{' '.join(prep_cmd)}

### (Optional) Compute attribute stats and quantization bins

Only needed when your training config has non-empty `CONTINUOUS_KEYS`. Writes `metadata_attributes.json`, `bins_values.pkl`, and `lmdb_keys.pkl` (NaN-free key list) inside `PROCESSED_LMDB`.

In [ ]:
# Uncomment to run.
# !{PYTHON} {REPO_PATH}/scripts/compute_min_max_dataset.py --data_path {PROCESSED_LMDB}

## Start initial training

Training output goes to `/kaggle/working/runs/<RUN_NAME>/` — checkpoints (`last.ckpt`, `best.ckpt`), TensorBoard logs, and a `config.gin` snapshot of the operative gin configuration.

***For initial training, the cell below is enabled. Disable it (and the preprocess section) when you want to resume training in a later session.***

In [ ]:
%cd /kaggle/working

train_cmd = [
    PYTHON, f'{REPO_PATH}/scripts/train.py',
    '-d', PROCESSED_LMDB,
    '-n', RUN_NAME,
    '-c', TRAIN_CONFIG,
    '-s', SAVE_DIR,
    '--gpu', '0',
    '--max_steps', str(MAX_STEPS),
    '--val_every', str(VAL_EVERY),
]
if BUILD_CACHE:      train_cmd.append('--build_cache')
if LMDB_KEYS_FILE:   train_cmd += ['--lmdb_keys_filename', LMDB_KEYS_FILE]
if BINS_VALUES_FILE: train_cmd += ['--bins_values_file',   BINS_VALUES_FILE]
if MIN_MAX_FILE:     train_cmd += ['--min_max_file',       MIN_MAX_FILE]

print(' '.join(train_cmd))
!{' '.join(train_cmd)}

## Resume training

Kaggle GPU sessions cap at ~12 hours, so long PLaTune runs need to be split across sessions. The pattern:

1. After a session finishes, in this notebook's **Output** tab click **Save Version → Save output as Dataset**.
2. Create a **new** notebook (or restart this one), attach that dataset as input, and set `RESUME_FROM_INPUT` in the Configuration cell to its mount path (e.g. `/kaggle/input/platune-run-v1`).
3. **Disable** the Preprocess and Initial-training sections above (also build the LMDB once and save it as a Dataset; attach it back so you don't redo preprocessing).
4. Run the cell below — it copies the prior run back into `/kaggle/working/`, then re-launches `train.py` with `--ckpt`.

Keep `TRAIN_CONFIG`, `RUN_NAME`, and any `--bins_values_file` / `--min_max_file` flags identical to the initial run.

In [ ]:
if RESUME_FROM_INPUT is None:
    raise RuntimeError('Set RESUME_FROM_INPUT in the Configuration cell to your previous run dataset mount path.')

# Copy previous run output (runs/<RUN_NAME>/...) back into /kaggle/working/
!cp -rn {RESUME_FROM_INPUT}/runs /kaggle/working/ 2>/dev/null || true
%cd /kaggle/working
!ls {SAVE_DIR}/{RUN_NAME}

resume_cmd = [
    PYTHON, f'{REPO_PATH}/scripts/train.py',
    '-d', PROCESSED_LMDB,
    '-n', RUN_NAME,
    '-c', TRAIN_CONFIG,
    '-s', SAVE_DIR,
    '--gpu', '0',
    '--max_steps', str(MAX_STEPS),
    '--val_every', str(VAL_EVERY),
    '--ckpt', f'{SAVE_DIR}/{RUN_NAME}',   # train.py recursively finds *last*.ckpt
]
if BUILD_CACHE:      resume_cmd.append('--build_cache')
if LMDB_KEYS_FILE:   resume_cmd += ['--lmdb_keys_filename', LMDB_KEYS_FILE]
if BINS_VALUES_FILE: resume_cmd += ['--bins_values_file',   BINS_VALUES_FILE]
if MIN_MAX_FILE:     resume_cmd += ['--min_max_file',       MIN_MAX_FILE]

print(' '.join(resume_cmd))
!{' '.join(resume_cmd)}

## Inspect outputs / TensorBoard

Lists everything written under `/kaggle/working/runs/<RUN_NAME>/` (this is what's preserved when the kernel commits) and renders TensorBoard inline.

In [ ]:
run_dir = os.path.join(SAVE_DIR, RUN_NAME)
for root, _, files in os.walk(run_dir):
    rel = os.path.relpath(root, run_dir)
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(root, f)) / 1024**2
        print(f'{rel}/{f}  ({size_mb:.1f} MB)')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {SAVE_DIR}/{RUN_NAME}